## Standard Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Preprocessing data

### Three main things we have to do:
1. Split the data into lables (`X` & `y`)
2. Filling (also called imputing) or disregarding missing values
3. Converting non-numerical to numerical values (feature encoding)

In [3]:
df = pd.read_csv("data/heart-disease.csv")
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [7]:
target_column = "target"
X = df.drop(target_column, axis=1)
y = df[target_column]
X.shape, y.shape

((303, 13), (303,))

 ## 1. Split data into testing and training sets

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

### 1.1 Make sure its all numerical

Well the heart-deases data set is already numerical. Let's load a dataset with some text values

In [11]:
df = pd.read_csv("data/car-sales-extended.csv")
df.head()

,Make,Colour,Odometer (KM),Doors,Price
0,Honda,White,35431,4,15323
1,BMW,Blue,192714,5,19943
2,Honda,White,84714,4,28343
3,Toyota,White,154365,4,13434
4,Nissan,Blue,181577,3,14043


##### Observation: Let's try to predict `Price`. 
As it is a numeric column, it will be a <b>regression problem</b>

In [12]:
df.shape

(1000, 5)

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Make           1000 non-null   object
 1   Colour         1000 non-null   object
 2   Odometer (KM)  1000 non-null   int64 
 3   Doors          1000 non-null   int64 
 4   Price          1000 non-null   int64 
dtypes: int64(3), object(2)
memory usage: 39.2+ KB


Let's produce an error by trying to train a model without turning texts into numbers aka encoding

In [20]:
# setting features and labels
target_var = "Price"
X = df.drop(target_var, axis=1)
y = df[target_var]

# split the sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Fit the model. Let's do a RandomForestRegression
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor()
model.fit(X_train, y_train)

# Evaluate the model
model.score(X_test, y_test)

ValueError: could not convert string to float: 'Toyota'

As we can see, we need to convert strings to numbers.

This method is called feature encoding

In [22]:
# Feature encoding
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ["Make", "Colour", "Doors"]

 yes, we are using Doors too! Because, The number of doors can be at most two to five types 

In [24]:
df["Doors"].value_counts()

4    856
5     79
3     65
Name: Doors, dtype: int64

So, there will be either 4, 5 or 3 doors in each car

<b>One Hot Encoding</b> can be defined as a process of transforming categorical variables into numerical format before fitting and training a Machine Learning algorithm.

![One Hot Encoding](https://www.statology.org/wp-content/uploads/2021/09/oneHot1.png)

<b>Column Transformer</b> is a scikit-learn class that applise transformers to pandas DataFrame (or columns of an array). This is useful for heterogeneous or columnar data, to combine several feature extraction mechanisms or transformations into a single transformer.

![Column Transformer](https://ubc-cs.github.io/cpsc330/_images/column-transformer.png)

<b>Call signature:</b>
```python
class sklearn.compose.ColumnTransformer(transformers, *, remainder='drop', sparse_threshold=0.3, n_jobs=None, transformer_weights=None, verbose=False, verbose_feature_names_out=True)
```

Here, `transformers` is a list of tuples.
List of <b><i>`(name, transformer, columns)`</i></b> tuples specifying the transformer objects to be applied to subsets of the data. where,

<b>name: str</b> a name for future reference<br/>
<b>transformer:</b> {`drop`, `passthrough`} or estimator class<br/>
<b>columns: str, array-like of str, int, slice etc.</b> Index of dataframe or similar

In [26]:
one_hot = OneHotEncoder()
transformer = ColumnTransformer(
                transformers=[("one_hot", one_hot, categorical_features)],
                remainder="passthrough")

# Apply the transformer on features X
transformed_X = transformer.fit_transform(X)

array([[0.00000e+00, 1.00000e+00, 0.00000e+00, ..., 1.00000e+00,
        0.00000e+00, 3.54310e+04],
       [1.00000e+00, 0.00000e+00, 0.00000e+00, ..., 0.00000e+00,
        1.00000e+00, 1.92714e+05],
       [0.00000e+00, 1.00000e+00, 0.00000e+00, ..., 1.00000e+00,
        0.00000e+00, 8.47140e+04],
       ...,
       [0.00000e+00, 0.00000e+00, 1.00000e+00, ..., 1.00000e+00,
        0.00000e+00, 6.66040e+04],
       [0.00000e+00, 1.00000e+00, 0.00000e+00, ..., 1.00000e+00,
        0.00000e+00, 2.15883e+05],
       [0.00000e+00, 0.00000e+00, 0.00000e+00, ..., 1.00000e+00,
        0.00000e+00, 2.48360e+05]])

In [28]:
# a bit clearer visualization
pd.DataFrame(transformed_X).head()

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,35431.0
1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,192714.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,84714.0
3,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,154365.0
4,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,181577.0


In [38]:
# Similar implementation with pandas get_dummies method
pd.get_dummies(X, columns=categorical_features).head()

,Odometer (KM),Make_BMW,Make_Honda,Make_Nissan,Make_Toyota,Colour_Black,Colour_Blue,Colour_Green,Colour_Red,Colour_White,Doors_3,Doors_4,Doors_5
0,35431,0,1,0,0,0,0,0,0,1,0,1,0
1,192714,1,0,0,0,0,1,0,0,0,0,0,1
2,84714,0,1,0,0,0,0,0,0,1,0,1,0
3,154365,0,0,0,1,0,0,0,0,1,0,1,0
4,181577,0,0,1,0,0,1,0,0,0,1,0,0


In [35]:
# compare with actual X
X.head()

,Make,Colour,Odometer (KM),Doors
0,Honda,White,35431,4
1,BMW,Blue,192714,5
2,Honda,White,84714,4
3,Toyota,White,154365,4
4,Nissan,Blue,181577,3


So, now all are data are in numbers. Let's retry and refit the model!

In [47]:
# test_train split
X_train, X_test, y_train, y_test = train_test_split(transformed_X, y, test_size=0.2)

# Let's choose the Random Forest Regressor again
model = RandomForestRegressor()
model.fit(X_train, y_train)

# Evaluate the model
model.score(X_test, y_test)

0.417628391779279

## 2. Handling missing data

1. Fill them with some value (also know as imputation)
2. Remove the sample with missing data altogether.

In [48]:
# Let's load data!
df = pd.read_csv('data/car-sales-extended-missing-data.csv')
df.head()

,Make,Colour,Odometer (KM),Doors,Price
0,Honda,White,35431.0,4.0,15323.0
1,BMW,Blue,192714.0,5.0,19943.0
2,Honda,White,84714.0,4.0,28343.0
3,Toyota,White,154365.0,4.0,13434.0
4,Nissan,Blue,181577.0,3.0,14043.0


In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Make           951 non-null    object 
 1   Colour         950 non-null    object 
 2   Odometer (KM)  950 non-null    float64
 3   Doors          950 non-null    float64
 4   Price          950 non-null    float64
dtypes: float64(3), object(2)
memory usage: 39.2+ KB


So, looks like there are some missing data.
Let's explore a little more.

In [56]:
df.isna().sum()

Make             49
Colour           50
Odometer (KM)    50
Doors            50
Price            50
dtype: int64

 <b>Note:</b> `isna()` and `isnull()` are actually same.
 `isnull()` is a method of Pandas Series objects, 
 while `isna()` is a method of both Series and DataFrame objects
 so, `isna()` would be a preferred choice.

Anyways, as there are some missing values in the set,
how do we proceed?

#### Option-1: Fill missing data with Pandas
For the <i>categorical</i> columns, we will fill the values with string: `missing`.<br>
For the <i>numerical values</i> we may use `mean` or similar statistics.

In [74]:
# Fill the "Make" column
df["Make"].fillna("missing", inplace=True)

# Fill the "Colour" column
df["Colour"].fillna("missing", inplace=True)

# Fill the "Odometer (KM)" column
avg_odometer_distance = df["Odometer (KM)"].mean()
df["Odometer (KM)"].fillna(avg_odometer_distance, inplace=True)

# Fill the "Doors" column
# Well, Doors column is a categorical feature. So we will fill it with the 
# `mode` in this case. the statistical term `mode` means the most occuring value.
most_frequent_number_of_doors = df["Doors"].mode()[0]
df["Doors"].fillna(most_frequent_number_of_doors, inplace=True)

In [75]:
# Let's check the missing values again
df.isna().sum()

Make              0
Colour            0
Odometer (KM)     0
Doors             0
Price            50
dtype: int64

Now, there are 50 prices that we don't know. As these price is the variable we are
trying to predict, we have no other option but to dorp it.

In [79]:
df.dropna(inplace=True)

In [81]:
# Let's check our current status of the data
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 950 entries, 0 to 999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Make           950 non-null    object 
 1   Colour         950 non-null    object 
 2   Odometer (KM)  950 non-null    float64
 3   Doors          950 non-null    float64
 4   Price          950 non-null    float64
dtypes: float64(3), object(2)
memory usage: 44.5+ KB


Ok, so there are no missing values in the features. 
We can also see that, the size of our data set is reduced to 950 from 1000 rows.
Which means all the null values in Price column (target variable) is dropped.

So now we can proceed to next steps

In [87]:
# Feature and label declaration
target_variable = "Price"
X = df.drop(target_variable, axis=1)
y = df[target_variable]

But wait, there are strings in our dataframe. So, we need to do some encoding first.
We'll do the same as before:

    1. Use one hot encoder to take care of Make, Colur and Doors
    2. Then, transform everything to our dataframe at once, with the help of scikit-learn `ColumnTransformer`

In [93]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Make the transformer
categorical_columns = ["Make", "Colour", "Doors"]

onehot_enc = OneHotEncoder()
column_transformer = ColumnTransformer(
                        transformers=[("onehot_enc", onehot_enc, categorical_columns)],
                        remainder="passthrough"
                    )

# Apply the transformer on X
transformed_X = column_transformer.fit_transform(X)

#### Option-2: Fill missing data with Scikit-Learn
So, what we did with pandas before, was actually called imputing, meaning we filled our missing values following some sort of strategies.<br>
However, Scikit-Learn also provides a simialr functionality, which is called Imputer.
The plus point is, we can directly combine the Imputer object to the ColumnTransformer class.

In [133]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

df = pd.read_csv('data/car-sales-extended-missing-data.csv')
target_variable = "Price"

# We can't take the rows, where there is no Price. So, drop it!
df = df.dropna(subset=target_variable)

X = df.drop(target_variable, axis=1)
y = df[target_variable]

# Fill the categorical values with 'missing' & numerical values with mean or mode
categorical_imputer = SimpleImputer(strategy="constant", fill_value="missing")

most_frequent_number_of_doors = df["Doors"].mode()[0]
door_imputer = SimpleImputer(strategy="constant", fill_value=4)

odometer_imputer = SimpleImputer(strategy="mean")

# Define the columns
categorical_features = ["Make", "Colour"]
door_features = ["Doors"]
odometer_features = ["Odometer (KM)"]

# Create a transformer to fill the X
impute_transformer = ColumnTransformer(
    transformers=[
        ("categorical_imputer", categorical_imputer, categorical_features),
        ("door_imputer", door_imputer, door_features),
        ("odometer_imputer", odometer_imputer, odometer_features),
    ],
    remainder="passthrough"
)

filled_X = impute_transformer.fit_transform(X)
filled_X

array([['Honda', 'White', 4.0, 35431.0],
       ['BMW', 'Blue', 5.0, 192714.0],
       ['Honda', 'White', 4.0, 84714.0],
       ...,
       ['Nissan', 'Blue', 4.0, 66604.0],
       ['Honda', 'White', 4.0, 215883.0],
       ['Toyota', 'Blue', 4.0, 248360.0]], dtype=object)

In [134]:
# let's just make it a data frame.
X_filled = pd.DataFrame(filled_X, columns=["Make", "Colour", "Doors", "Odometer (KM)"])
X_filled.head()

,Make,Colour,Doors,Odometer (KM)
0,Honda,White,4.0,35431.0
1,BMW,Blue,5.0,192714.0
2,Honda,White,4.0,84714.0
3,Toyota,White,4.0,154365.0
4,Nissan,Blue,3.0,181577.0


In [135]:
X_filled.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 950 entries, 0 to 949
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Make           950 non-null    object
 1   Colour         950 non-null    object
 2   Doors          950 non-null    object
 3   Odometer (KM)  950 non-null    object
dtypes: object(4)
memory usage: 29.8+ KB


Now, Let the encoding begin!

In [137]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

onehot_enc = OneHotEncoder()
categorical_features = ["Make", "Colour", "Doors"]

onehot_transformer = ColumnTransformer(transformers=[("onehot_enc", onehot_enc, categorical_features)], remainder="passthrough")

X_transformed = onehot_transformer.fit_transform(X_filled)
X_transformed

<950x15 sparse matrix of type '<class 'numpy.float64'>'
	with 3800 stored elements in Compressed Sparse Row format>

Let the training begin

In [164]:

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_transformed, y, test_size=0.2)

# Fit to a model
model = RandomForestRegressor()
model.fit(X_train, y_train)

# Evaluation
model.score(X_test, y_test)

0.23243492420386014